In [ ]:
# ================================================================
# CELL 1 — INSTALLS & IMPORTS
# ================================================================
import os, glob, random, copy, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score

print('Imports done.')

# ---- GPU / device setup ----
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — running on CPU.')


In [ ]:
dataset_names = ['Compas','Compas','German','Bank']
protected_attributes = ['sex','race','sex','age']

CSV_DIR = r'/home/shivampathak/Desktop/New Folder/preprocessed_data'      # <-- EDIT: folder containing *_processed.csv
RESULTS_DIR = r'results_FederatedLearning_new_tist'      # <-- EDIT: ONE shared results folder for ALL 11 combos
TARGET_COL_DEFAULT = 'Probability'         # change if a dataset uses a different name
TARGET_COL_OVERRIDES = {}                  # e.g. {'Heart': 'target'}

os.makedirs(RESULTS_DIR, exist_ok=True)
MASTER_LOG_PATH   = os.path.join(RESULTS_DIR, 'master_log.csv')
HISTORY_LOG_PATH  = os.path.join(RESULTS_DIR, 'round_history_log.csv')
ROC_DIR           = os.path.join(RESULTS_DIR, 'roc_data')      # per-run test-set y_true/y_score/protected, for AUC/ROC plots later — no retraining needed
os.makedirs(ROC_DIR, exist_ok=True)

# ---- Fixed pipeline hyperparameters (methodology identical to main pipeline) ----
NUM_CLIENTS      = 10
LOCAL_EPOCHS     = 5
BATCH_SIZE       = 32
LR               = 0.001
TEST_SIZE        = 0.20

DEFAULT_ALPHA    = 1.5
DEFAULT_BETA     = 3.0
DEFAULT_EMA      = 0.7
DEFAULT_MU       = 0.5
DEFAULT_EPS_FAIR = 0.1

WGAN_EPOCHS  = 300
WGAN_LR      = 1e-4
N_CRITIC     = 5
CLIP_VALUE   = 0.01
WGAN_HIDDEN  = 64
WGAN_LATENT  = 32

WASSERSTEIN_QUANTILES = 20   # NEW: quantile points used by the differentiable local fairness penalty (soft_wasserstein_penalty)

# ---- Experiment sizing (increase once the pipeline is validated end to end) ----
NUM_ROUNDS_MAIN   = 30
NUM_ROUNDS_SWEEP  = 20

SEEDS_MAIN     = [42, 43, 44, 45, 46]
SEEDS_SWEEP    = [42, 43, 44]
SEEDS_ABLATION = [42, 43, 44, 45, 46]

ALPHA_VALUES  = [0.5, 1.0, 1.5, 2.0, 3.0]
BETA_VALUES   = [0.5, 1.0, 2.0, 3.0, 5.0]
LAMBDA_VALUES = [0.3, 0.5, 0.7, 0.8, 0.9]
MU_VALUES     = [0.1, 0.3, 0.5, 0.7, 1.0]     # NEW — mu now actually has a live gradient effect, so it is worth sweeping

# indices into dataset_names/protected_attributes used for the ablation study
ABLATION_DATASET_IDX = [0,1,2,3]     # pick representative dataset-attribute combo(s), e.g. Compas-sex
ABLATION_VARIANTS = ['full', 'no_wgan', 'no_ema', 'fedavg_agg', 'final_round_only', 'no_fair_penalty']
# NEW: 'no_fair_penalty' isolates the effect of the corrected differentiable local
# fairness term — full vs no_fair_penalty is the cleanest evidence that the fix matters.

SAVE_MODEL_WEIGHTS = False   # True -> also pickle final selected weights per run (slow/large)
SAVE_ROC_DATA       = True   # NEW: True -> save y_true / y_score / protected for the final selected model of
                              # each MAIN run (small — one float array per run) so ROC/AUC curves can be
                              # redrawn later WITHOUT rerunning training.

RUN_MAIN        = True
RUN_SENSITIVITY = True
RUN_ABLATION    = True

print('Config set.')
print(f'{len(dataset_names)} dataset-attribute combinations queued.')


In [ ]:
class WGANGenerator(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),   nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2),   nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, output_dim), nn.Sigmoid()
        )
    def forward(self, z): return self.net(z)


class WGANCritic(nn.Module):
    def __init__(self, input_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden * 2), nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, hidden),    nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1)
        )
    def forward(self, x): return self.net(x)


def train_wgan(real_data_np, latent_dim=32, hidden=64, epochs=300, lr=1e-4,
               n_critic=5, clip=0.01, batch_size=32, seed=42, device=None):
    device = device or DEVICE
    torch.manual_seed(seed); np.random.seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)
    n, feat_dim = real_data_np.shape
    if n < 5:
        return None
    loader = DataLoader(TensorDataset(torch.tensor(real_data_np, dtype=torch.float32)),
                         batch_size=min(batch_size, n), shuffle=True, drop_last=False)
    G = WGANGenerator(latent_dim, feat_dim, hidden).to(device)
    C = WGANCritic(feat_dim, hidden).to(device)
    opt_G = optim.RMSprop(G.parameters(), lr=lr)
    opt_C = optim.RMSprop(C.parameters(), lr=lr)
    G.train(); C.train()
    for _ in range(epochs):
        for (rb,) in loader:
            rb = rb.to(device)
            bs = rb.size(0)
            if bs < 2:
                continue
            for _ in range(n_critic):
                z = torch.randn(bs, latent_dim, device=device)
                loss_C = -(C(rb).mean() - C(G(z).detach()).mean())
                opt_C.zero_grad(); loss_C.backward(); opt_C.step()
                for p in C.parameters(): p.data.clamp_(-clip, clip)
            loss_G = -C(G(torch.randn(bs, latent_dim, device=device))).mean()
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    G.eval()
    return G


def generate_synthetic(generator, n_samples, latent_dim, real_data_np,
                        col_names, target_val, protected_val, target_col, protected_col, device=None):
    device = device or DEVICE
    if n_samples <= 0:
        return pd.DataFrame(columns=col_names)
    if generator is None:
        idx = np.random.choice(len(real_data_np), n_samples, replace=True)
        synth = np.clip(real_data_np[idx] + np.random.normal(0, 0.01, real_data_np[idx].shape), 0, 1)
    else:
        with torch.no_grad():
            synth = generator(torch.randn(n_samples, latent_dim, device=device)).cpu().numpy()
    feat_cols = [c for c in col_names if c not in [target_col, protected_col]]
    df_s = pd.DataFrame(synth, columns=feat_cols)
    df_s[target_col] = target_val
    df_s[protected_col] = protected_val
    return df_s[col_names]


class GlobalMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, 1),         nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)



def soft_wasserstein_penalty(scores, protected_batch, num_quantiles=20):
    """
    Differentiable 1D Wasserstein-distance surrogate between the model's
    sigmoid outputs of the two protected groups within a batch.

    scores            : (B,) tensor of model outputs in [0,1], part of the
                         autograd graph (i.e. NOT detached).
    protected_batch   : (B,) tensor of 0/1 group membership (no grad needed).
    num_quantiles     : number of quantile points used for matching.

    Returns a 0-d tensor, differentiable wrt `scores`. Uses quantile matching
    (not naive sorted-pairing) so it handles unequal group sizes correctly —
    this matters here because local batches, even after WGAN balancing, can
    still have skewed group counts.
    """
    mask0 = protected_batch == 0
    mask1 = protected_batch == 1
    n0 = int(mask0.sum().item())
    n1 = int(mask1.sum().item())
    if n0 == 0 or n1 == 0:
        # Nothing to compare against in this batch — skip rather than error.
        return torch.zeros((), device=scores.device, dtype=scores.dtype)
    m = max(2, min(num_quantiles, n0, n1))
    qs = torch.linspace(0.0, 1.0, steps=m, device=scores.device)
    q0 = torch.quantile(scores[mask0], qs)
    q1 = torch.quantile(scores[mask1], qs)
    return torch.mean(torch.abs(q0 - q1))


def fairness_metrics(y_true, y_pred, protected, y_score=None):
    """
    Extensive fairness/performance metric dict.
    Returns: DI, SPD, EOD, AOD, FPRD, TNRD, PPD (group-gap metrics), plus
    per-group TPR/FPR/TNR/FNR/precision/F1, overall F1/precision, and
    AUC (overall + per-group) when y_score is supplied.
    signed DI = 1 - p_unpriv/p_priv (ideal = 0).
    """
    y_true = np.array(y_true); y_pred = np.array(y_pred); protected = np.array(protected)
    priv_idx = protected == 1
    unpriv_idx = protected == 0
    p_priv   = np.mean(y_pred[priv_idx] == 1)   if priv_idx.sum()   > 0 else 0.0
    p_unpriv = np.mean(y_pred[unpriv_idx] == 1) if unpriv_idx.sum() > 0 else 0.0
    SPD = p_unpriv - p_priv
    DI  = 1.0 - (p_unpriv / p_priv) if p_priv > 1e-9 else np.nan

    def rates(yt, yp):
        if len(yt) == 0:
            return dict(tpr=0.0, fpr=0.0, tnr=0.0, fnr=0.0, precision=0.0, f1=0.0, acc=0.0)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0.0
        acc_g = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        return dict(tpr=tpr, fpr=fpr, tnr=tnr, fnr=fnr, precision=precision, f1=f1, acc=acc_g)

    r_priv    = rates(y_true[priv_idx],   y_pred[priv_idx])
    r_unpriv  = rates(y_true[unpriv_idx], y_pred[unpriv_idx])
    r_overall = rates(y_true, y_pred)

    EOD  = r_unpriv['tpr'] - r_priv['tpr']
    AOD  = ((r_unpriv['fpr'] - r_priv['fpr']) + (r_unpriv['tpr'] - r_priv['tpr'])) / 2
    FPRD = r_unpriv['fpr'] - r_priv['fpr']         # predictive-equality gap
    TNRD = r_unpriv['tnr'] - r_priv['tnr']
    PPD  = r_unpriv['precision'] - r_priv['precision']   # predictive-parity gap

    out = {
        'DI': DI, 'SPD': SPD, 'EOD': EOD, 'AOD': AOD, 'FPRD': FPRD, 'TNRD': TNRD, 'PPD': PPD,
        'tpr_priv': r_priv['tpr'],   'tpr_unpriv': r_unpriv['tpr'],
        'fpr_priv': r_priv['fpr'],   'fpr_unpriv': r_unpriv['fpr'],
        'tnr_priv': r_priv['tnr'],   'tnr_unpriv': r_unpriv['tnr'],
        'fnr_priv': r_priv['fnr'],   'fnr_unpriv': r_unpriv['fnr'],
        'precision_priv': r_priv['precision'], 'precision_unpriv': r_unpriv['precision'],
        'f1_priv': r_priv['f1'],     'f1_unpriv': r_unpriv['f1'],
        'f1_overall': r_overall['f1'], 'precision_overall': r_overall['precision'],
    }
    if y_score is not None:
        y_score = np.array(y_score)
        try:
            out['auc_overall'] = roc_auc_score(y_true, y_score)
        except ValueError:
            out['auc_overall'] = np.nan
        try:
            out['auc_priv'] = (roc_auc_score(y_true[priv_idx], y_score[priv_idx])
                                if priv_idx.sum() > 0 and len(np.unique(y_true[priv_idx])) > 1 else np.nan)
        except ValueError:
            out['auc_priv'] = np.nan
        try:
            out['auc_unpriv'] = (roc_auc_score(y_true[unpriv_idx], y_score[unpriv_idx])
                                  if unpriv_idx.sum() > 0 and len(np.unique(y_true[unpriv_idx])) > 1 else np.nan)
        except ValueError:
            out['auc_unpriv'] = np.nan
    return out


def full_eval(model, X_tensor, y_np, prot_np, return_scores=False):
    """NEW: can also return raw sigmoid scores (probs) so AUC/ROC curves
    can be drawn later without retraining."""
    model.eval()
    with torch.no_grad():
        probs = model(X_tensor).squeeze().cpu().numpy()
    preds = (probs >= 0.5).astype(float)
    acc  = accuracy_score(y_np, preds)
    bacc = balanced_accuracy_score(y_np, preds)
    fm   = fairness_metrics(y_np.astype(int), preds.astype(int), prot_np, y_score=probs)
    if return_scores:
        return acc, bacc, fm, probs
    return acc, bacc, fm


def is_pareto_nondominated(costs):
    """costs: (N,M), lower=better. Returns bool mask of non-dominated rows."""
    N = costs.shape[0]
    dominated = np.zeros(N, dtype=bool)
    for i in range(N):
        for j in range(N):
            if i == j: continue
            if np.all(costs[j] <= costs[i]) and np.any(costs[j] < costs[i]):
                dominated[i] = True
                break
    return ~dominated

print('Model / WGAN / metric definitions ready (extended metrics + differentiable Wasserstein penalty).')


In [ ]:
def run_pipeline(df, protected_attr, target_col,
                  alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA, ema_lambda=DEFAULT_EMA,
                  mu=DEFAULT_MU, eps_fair=DEFAULT_EPS_FAIR,
                  num_clients=NUM_CLIENTS, num_rounds=NUM_ROUNDS_MAIN,
                  local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                  seed=42, variant='full'):
    """
    variant in {'full','no_wgan','no_ema','fedavg_agg','final_round_only','no_fair_penalty'}
      - 'no_fair_penalty' is NEW: skips the local fairness penalty entirely,
        isolating the effect of the (now-corrected) differentiable term.

    Data split:
      - 70% training data: used for client-local training and WGAN balancing.
      - 15% validation data: used for ALL per-round evaluation, EMA-DI,
        aggregation weighting, and candidate/model selection.
      - 15% test data: NEVER used during federated training or model selection.
        It is used only once, after the winning client/global model has been
        selected, to report the final unbiased test metrics and probabilities.

    Returns a dict with the Pareto-selected model's FINAL TEST metrics
    (Acc, BAcc, DI-derived 1-DI, SPD, EOD, AOD, plus the extended
    TPR/FPR/TNR/FNR/F1/AUC metrics), a per-round validation history, and the
    winning model's raw TEST-set probabilities ('final_probs') so ROC/AUC
    curves can be drawn later without retraining.
    """
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(seed)
    EPS_NUM = 1e-9

    # ------------------------------------------------------------------
    # 70 / 15 / 15 split:
    #   train      -> local client training + WGAN balancing
    #   validation -> per-round evaluation, aggregation weighting,
    #                 EMA-DI and candidate/model selection
    #   test       -> FINAL evaluation ONLY after model selection
    # ------------------------------------------------------------------
    train_df, temp_df = train_test_split(
        df, test_size=0.30, random_state=seed, shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=seed, shuffle=True
    )

    train_df = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    # NOTE: np.array_split(DataFrame, n) silently returns raw numpy arrays (not DataFrame
    # chunks) on some numpy/pandas version combinations, which breaks `.columns` downstream.
    # Splitting on the index instead guarantees DataFrame chunks regardless of version.
    client_idx_splits = np.array_split(np.arange(len(train_df)), num_clients)
    clients_raw = [train_df.iloc[idx].reset_index(drop=True) for idx in client_idx_splits]

    # Validation set: this is the ONLY held-out set used during rounds.
    X_val_np = val_df.drop(columns=[target_col]).values.astype(np.float32)
    y_val_np = val_df[target_col].values.astype(np.float32)
    prot_val = val_df[protected_attr].values
    X_val_tensor = torch.tensor(X_val_np).to(DEVICE)

    # Test set: kept completely untouched until final model selection.
    X_test_np = test_df.drop(columns=[target_col]).values.astype(np.float32)
    y_test_np = test_df[target_col].values.astype(np.float32)
    prot_test = test_df[protected_attr].values
    X_test_tensor = torch.tensor(X_test_np).to(DEVICE)

    input_dim = X_val_np.shape[1]

    all_cols = clients_raw[0].columns.tolist()
    feat_cols_nn = [c for c in all_cols if c not in [target_col, protected_attr]]

    # ---- Phase I: WGAN subgroup balancing (skipped for 'no_wgan' ablation) ----
    if variant == 'no_wgan':
        balanced_clients = [c.reset_index(drop=True) for c in clients_raw]
    else:
        balanced_clients = []
        for cid, cdf in enumerate(clients_raw):
            subgroups = {(t, p): cdf[(cdf[target_col] == t) & (cdf[protected_attr] == p)]
                                   .copy().reset_index(drop=True)
                         for t in [0, 1] for p in [0, 1]}
            max_size = max(len(v) for v in subgroups.values())
            augmented = []
            for (t, p), sub_df in subgroups.items():
                deficit = max_size - len(sub_df)
                if deficit > 0:
                    real_np = sub_df[feat_cols_nn].values.astype(np.float32)
                    gen = train_wgan(real_np, latent_dim=WGAN_LATENT, hidden=WGAN_HIDDEN,
                                      epochs=WGAN_EPOCHS, lr=WGAN_LR, n_critic=N_CRITIC,
                                      clip=CLIP_VALUE, seed=seed + cid*10 + t*2 + p, device=DEVICE)
                    synth_df = generate_synthetic(gen, deficit, WGAN_LATENT, real_np, all_cols,
                                                   t, p, target_col, protected_attr, device=DEVICE)
                    sub_df = pd.concat([sub_df, synth_df], ignore_index=True)
                augmented.append(sub_df)
            bal_df = pd.concat(augmented, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
            bal_df[target_col] = bal_df[target_col].round().astype(int)
            bal_df[protected_attr] = bal_df[protected_attr].round().astype(int)
            balanced_clients.append(bal_df)

    # ---- Phase II: Federated training + candidate archiving ----
    global_model = GlobalMLP(input_dim).to(DEVICE)
    bce_loss = nn.BCELoss()
    ema_di = {i: 0.0 for i in range(num_clients)}   # still used for aggregation weighting + best-client selection ONLY
    candidate_pool = []
    round_history = []

    for rnd in range(1, num_rounds + 1):
        local_weights, round_acc, round_bacc, round_probs = [], [], [], []
        round_di_signed, round_di_ema, round_fm, client_n = [], [], [], []

        for cid in range(num_clients):
            local_model = GlobalMLP(input_dim).to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())
            local_model.train()
            opt = optim.Adam(local_model.parameters(), lr=lr)

            bal_df = balanced_clients[cid]
            X_bal = torch.tensor(bal_df.drop(columns=[target_col]).values.astype(np.float32))
            y_bal = torch.tensor(bal_df[target_col].values.astype(np.float32))
            prot_bal = torch.tensor(bal_df[protected_attr].values.astype(np.float32))   # NEW: needed by soft_wasserstein_penalty
            loader = DataLoader(TensorDataset(X_bal, y_bal, prot_bal), batch_size=batch_size,
                                 shuffle=True, drop_last=False)

            for _ in range(local_epochs):
                for bX, by, bprot in loader:
                    if bX.size(0) < 2:
                        continue   # BatchNorm1d needs >=2 samples per batch during training
                    bX, by, bprot = bX.to(DEVICE), by.to(DEVICE), bprot.to(DEVICE)
                    opt.zero_grad()
                    out = local_model(bX).squeeze()
                    if out.dim() == 0: out = out.unsqueeze(0)
                    loss_ce = bce_loss(out, by)

                    
                    if variant == 'no_fair_penalty':
                        fair_penalty = torch.zeros((), device=DEVICE)
                    else:
                        w1_pen = soft_wasserstein_penalty(out, bprot, num_quantiles=WASSERSTEIN_QUANTILES)
                        fair_penalty = mu * torch.clamp(w1_pen - eps_fair, min=0.0)

                    (loss_ce + fair_penalty).backward()
                    opt.step()

           
            l_acc, l_bacc, l_fm, l_probs = full_eval(
                local_model, X_val_tensor, y_val_np, prot_val, return_scores=True
            )
            l_di = l_fm['DI']
            if np.isnan(l_di): l_di = ema_di[cid]

           
            new_ema = l_di if variant == 'no_ema' else (ema_lambda * ema_di[cid] + (1 - ema_lambda) * l_di)
            ema_di[cid] = new_ema

            round_acc.append(l_acc); round_bacc.append(l_bacc)
            round_di_signed.append(l_di); round_di_ema.append(new_ema)
            round_fm.append(l_fm)
            round_probs.append(l_probs)
            client_n.append(len(bal_df))
            local_weights.append(copy.deepcopy(local_model.state_dict()))

        # ---- Aggregation (UNCHANGED — never used gradients, so nothing here needed fixing) ----
        if variant == 'fedavg_agg':
            n_arr = np.array(client_n, dtype=np.float64)
            agg_weights = n_arr / n_arr.sum()
        else:
            raw_scores = np.array([
                (round_acc[i] ** alpha) * np.exp(-beta * abs(round_di_ema[i]))
                for i in range(num_clients)
            ], dtype=np.float64)
            agg_weights = raw_scores / (raw_scores.sum() + EPS_NUM)

        best_cid = int(np.argmin([abs(d) for d in round_di_signed]))
        best_fm = round_fm[best_cid]
        cand_client = {
            'label': f'Client{best_cid+1}_R{rnd}', 'type': 'client', 'round': rnd,
            'acc': round_acc[best_cid], 'bacc': round_bacc[best_cid],
            'di_abs': abs(round_di_signed[best_cid]), 'di_signed': round_di_signed[best_cid],
            'probs': round_probs[best_cid],
            
            'state_dict': {k: v.detach().cpu().clone()
                           for k, v in local_weights[best_cid].items()},
        }
        cand_client.update(best_fm)   # flatten extended metrics (spd/eod/aod/tpr_*/fpr_*/.../auc_*) in
        candidate_pool.append(cand_client)

        global_dict = global_model.state_dict()
        for key in global_dict:
            global_dict[key] = sum(agg_weights[i] * local_weights[i][key].float()
                                    for i in range(num_clients))
        global_model.load_state_dict(global_dict)

        # --------------------------------------------------------------
        # Global model is also evaluated on VALIDATION only.
        # --------------------------------------------------------------
        g_acc, g_bacc, g_fm, g_probs = full_eval(
            global_model, X_val_tensor, y_val_np, prot_val, return_scores=True
        )
        g_di = g_fm['DI'] if not np.isnan(g_fm['DI']) else 0.0
        cand_global = {
                'label': f'Global_R{rnd}', 'type': 'global', 'round': rnd,
                # These metrics are VALIDATION metrics used for candidate
                # selection. They are NOT final test metrics.
                'acc': g_acc, 'bacc': g_bacc,
                'di_abs': abs(g_di), 'di_signed': g_di,
                'probs': g_probs,
                'state_dict': {k: v.detach().cpu().clone()
                               for k, v in global_model.state_dict().items()},
            }
        cand_global.update(g_fm)
        candidate_pool.append(cand_global)

        round_history.append({
            'round': rnd,
            # ---- Global model this round (VALIDATION metrics) ----
            'global_acc': g_acc, 'global_bacc': g_bacc, 'global_di1': 1 - g_di,
            'global_spd': g_fm['SPD'], 'global_eod': g_fm['EOD'], 'global_aod': g_fm['AOD'],
            'global_auc': g_fm.get('auc_overall', np.nan),
            'mean_client_di1': float(np.mean([1 - d for d in round_di_signed])),
            # ---- Best client model this round (selected using VALIDATION) ----
            'best_client_id':   best_cid + 1,
            'best_client_acc':  round_acc[best_cid],
            'best_client_bacc': round_bacc[best_cid],
            'best_client_di1':  1 - round_di_signed[best_cid],
            'best_client_spd':  best_fm['SPD'], 'best_client_eod': best_fm['EOD'], 'best_client_aod': best_fm['AOD'],
            'best_client_auc':  best_fm.get('auc_overall', np.nan),
        })

    # ---- Phase III/IV: candidate selection (UNCHANGED) ----
    if variant == 'final_round_only':
        best_candidate = candidate_pool[-1]
    else:
        accs  = np.array([c['acc']    for c in candidate_pool])
        baccs = np.array([c['bacc']   for c in candidate_pool])
        dis   = np.array([c['di_abs'] for c in candidate_pool])
        cost_acc, cost_bacc = 1 - accs, 1 - baccs
        front_v1 = is_pareto_nondominated(np.column_stack([cost_acc, dis]))
        front_v2 = is_pareto_nondominated(np.column_stack([cost_bacc, dis]))
        front_v3 = is_pareto_nondominated(np.column_stack([cost_acc, cost_bacc, dis]))
        combined = front_v1 | front_v2 | front_v3
        composite = (accs ** alpha) * (baccs ** 0.5) * np.exp(-beta * dis)
        composite_masked = np.where(combined, composite, -np.inf)
        best_idx = int(np.argmax(composite_masked))
        best_candidate = candidate_pool[best_idx]

    result = {
        'selected_label':  best_candidate['label'],
        'selected_type':   best_candidate['type'],            # 'client' or 'global' — which one won
        'selected_round':  best_candidate['round'],
        'final_acc':       best_candidate['acc'],
        'final_bacc':      best_candidate['bacc'],
        'final_di1':       1 - best_candidate['di_signed'],   # PRIMARY metric, ideal = 1
        'final_spd':       best_candidate['SPD'],
        'final_eod':       best_candidate['EOD'],
        'final_aod':       best_candidate['AOD'],
        # ---- NEW: extended metrics of the selected model ----
        'final_fprd': best_candidate['FPRD'], 'final_tnrd': best_candidate['TNRD'], 'final_ppd': best_candidate['PPD'],
        'tpr_priv': best_candidate['tpr_priv'], 'tpr_unpriv': best_candidate['tpr_unpriv'],
        'fpr_priv': best_candidate['fpr_priv'], 'fpr_unpriv': best_candidate['fpr_unpriv'],
        'tnr_priv': best_candidate['tnr_priv'], 'tnr_unpriv': best_candidate['tnr_unpriv'],
        'fnr_priv': best_candidate['fnr_priv'], 'fnr_unpriv': best_candidate['fnr_unpriv'],
        'precision_priv': best_candidate['precision_priv'], 'precision_unpriv': best_candidate['precision_unpriv'],
        'f1_priv': best_candidate['f1_priv'], 'f1_unpriv': best_candidate['f1_unpriv'],
        'f1_overall': best_candidate['f1_overall'], 'precision_overall': best_candidate['precision_overall'],
        'auc_overall': best_candidate.get('auc_overall', np.nan),
        'auc_priv': best_candidate.get('auc_priv', np.nan),
        'auc_unpriv': best_candidate.get('auc_unpriv', np.nan),
        # ---- for later ROC/AUC plotting, no retraining needed ----
        'final_probs':     best_candidate['probs'],
        'y_test':          y_test_np,
        'prot_test':       prot_test,
        'round_history':   round_history,
    }
    if SAVE_MODEL_WEIGHTS:
        result['state_dict'] = best_candidate.get('state_dict')
    return result

print('run_pipeline() ready (differentiable local fairness penalty + extended metrics).')


In [ ]:
def get_csv_path(dataset_name):
    return os.path.join(CSV_DIR, f'{dataset_name.lower()}_processed.csv')


def load_dataset(dataset_name, protected_attr):
    path = get_csv_path(dataset_name)
    df = pd.read_csv(path)
    target_col = TARGET_COL_OVERRIDES.get(dataset_name, TARGET_COL_DEFAULT)
    feature_cols = [c for c in df.columns if c not in [target_col, protected_attr]]
    numeric_cols = df[feature_cols].select_dtypes(include=['int64', 'float64']).columns
    scaler = MinMaxScaler()
    df = df.copy()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df[target_col] = df[target_col].astype(int)
    df[protected_attr] = df[protected_attr].astype(int)
    return df, target_col


def _norm(v):
    return '' if v is None else str(v)


def load_completed_keys():
    if not os.path.exists(MASTER_LOG_PATH):
        return set()
    log = pd.read_csv(MASTER_LOG_PATH)
    keys = set(zip(
        log['dataset'], log['protected_attr'], log['experiment_type'], log['variant'],
        log['param_name'].fillna('').astype(str), log['param_value'].fillna('').astype(str),
        log['seed'].astype(str)
    ))
    return keys


def append_master_row(row: dict):
    df_row = pd.DataFrame([row])
    header = not os.path.exists(MASTER_LOG_PATH)
    df_row.to_csv(MASTER_LOG_PATH, mode='a', header=header, index=False)


def append_history_rows(dataset, protected_attr, experiment_type, variant,
                         param_name, param_value, seed, round_history):
    rows = []
    for rh in round_history:
        r = dict(rh)
        r.update({
            'dataset': dataset, 'protected_attr': protected_attr,
            'experiment_type': experiment_type, 'variant': variant,
            'param_name': _norm(param_name), 'param_value': _norm(param_value),
            'seed': seed,
        })
        rows.append(r)
    df_rows = pd.DataFrame(rows)
    header = not os.path.exists(HISTORY_LOG_PATH)
    df_rows.to_csv(HISTORY_LOG_PATH, mode='a', header=header, index=False)


def save_roc_data(dataset_label, protected_attr, seed, y_test, final_probs, prot_test,
                   alpha=None, beta=None, ema_lambda=None, mu=None, eps_fair=None,
                   selected_round=None, selected_type=None):
    """Persist the winning (Pareto-selected) model's test-set labels/scores/protected
    attribute for a MAIN run, so ROC/AUC curves and cross-method AUC comparisons can be
    generated later without ever rerunning training. Also embeds the hyperparameters
    and which candidate type won (client vs. global) so the file is self-describing
    even if separated from master_log.csv."""
    fname = f'{dataset_label}__seed{seed}.npz'
    np.savez_compressed(os.path.join(ROC_DIR, fname),
                         y_true=np.asarray(y_test), y_score=np.asarray(final_probs),
                         protected=np.asarray(prot_test),
                         alpha=alpha, beta=beta, ema_lambda=ema_lambda, mu=mu, eps_fair=eps_fair,
                         selected_round=selected_round, selected_type=selected_type)

print('Logging helpers ready.')


In [ ]:
completed = load_completed_keys()
ablation_targets = {(dataset_names[i], protected_attributes[i]) for i in ABLATION_DATASET_IDX}

def already_done(key):
    return key in completed

def mark_done(key):
    completed.add(key)

def make_key(dataset, protected_attr, experiment_type, variant, param_name, param_value, seed):
    return (dataset, protected_attr, experiment_type, variant,
            _norm(param_name), _norm(param_value), str(seed))


def _extended_row_fields(res):
    """NEW: pull every extended metric out of a run_pipeline() result dict
    so master_log.csv carries the full metric set in one pass — no need to
    ever rerun training just to compute one more metric later."""
    return {
        'fprd': res['final_fprd'], 'tnrd': res['final_tnrd'], 'ppd': res['final_ppd'],
        'tpr_priv': res['tpr_priv'], 'tpr_unpriv': res['tpr_unpriv'],
        'fpr_priv': res['fpr_priv'], 'fpr_unpriv': res['fpr_unpriv'],
        'tnr_priv': res['tnr_priv'], 'tnr_unpriv': res['tnr_unpriv'],
        'fnr_priv': res['fnr_priv'], 'fnr_unpriv': res['fnr_unpriv'],
        'precision_priv': res['precision_priv'], 'precision_unpriv': res['precision_unpriv'],
        'f1_priv': res['f1_priv'], 'f1_unpriv': res['f1_unpriv'],
        'f1_overall': res['f1_overall'], 'precision_overall': res['precision_overall'],
        'auc_overall': res['auc_overall'], 'auc_priv': res['auc_priv'], 'auc_unpriv': res['auc_unpriv'],
    }


for ds_name, prot_attr in zip(dataset_names, protected_attributes):
    ds_label = f'{ds_name}_{prot_attr}'
    print(f'\n=== {ds_label} ===')

    try:
        df, target_col = load_dataset(ds_name, prot_attr)
    except FileNotFoundError:
        print(f'  !! CSV not found for {ds_name} at {get_csv_path(ds_name)} — skipping.')
        continue

    # ---- Experiment 1: main comparison, default hyperparameters ----
    if RUN_MAIN:
        for seed in SEEDS_MAIN:
            key = make_key(ds_label, prot_attr, 'main', 'full', None, None, seed)
            if already_done(key):
                continue
            res = run_pipeline(df, prot_attr, target_col, num_rounds=NUM_ROUNDS_MAIN,
                                seed=seed, variant='full')
            row = {
                'dataset': ds_label, 'protected_attr': prot_attr,
                'experiment_type': 'main', 'method': 'Proposed', 'variant': 'full',
                'param_name': '', 'param_value': '', 'seed': seed,
                'selected_round': res['selected_round'], 'selected_type': res['selected_type'],
                'acc': res['final_acc'], 'bacc': res['final_bacc'], 'di1': res['final_di1'],
                'spd': res['final_spd'], 'eod': res['final_eod'], 'aod': res['final_aod'],
            }
            row.update(_extended_row_fields(res))
            append_master_row(row)
            append_history_rows(ds_label, prot_attr, 'main', 'full', None, None, seed, res['round_history'])
            if SAVE_ROC_DATA:
                save_roc_data(ds_label, prot_attr, seed, res['y_test'], res['final_probs'], res['prot_test'],
                               alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA, ema_lambda=DEFAULT_EMA,
                               mu=DEFAULT_MU, eps_fair=DEFAULT_EPS_FAIR,
                               selected_round=res['selected_round'], selected_type=res['selected_type'])
            mark_done(key)
            print(f'  [main] seed={seed}  Acc={res["final_acc"]:.4f}  1-DI={res["final_di1"]:.4f}  AUC={res["auc_overall"]:.4f}')

    # ---- Experiment 2: hyperparameter sensitivity ----
    if RUN_SENSITIVITY:
        sweep_specs = [
            ('alpha',  ALPHA_VALUES,  dict(beta=DEFAULT_BETA, ema_lambda=DEFAULT_EMA, mu=DEFAULT_MU)),
            ('beta',   BETA_VALUES,   dict(alpha=DEFAULT_ALPHA, ema_lambda=DEFAULT_EMA, mu=DEFAULT_MU)),
            ('lambda', LAMBDA_VALUES, dict(alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA, mu=DEFAULT_MU)),
            ('mu',     MU_VALUES,     dict(alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA, ema_lambda=DEFAULT_EMA)),  # NEW
        ]
        for param_name, values, fixed in sweep_specs:
            for val in values:
                for seed in SEEDS_SWEEP:
                    key = make_key(ds_label, prot_attr, 'sensitivity', 'full', param_name, val, seed)
                    if already_done(key):
                        continue
                    kwargs = dict(fixed)
                    kwargs['ema_lambda' if param_name == 'lambda' else param_name] = val
                    res = run_pipeline(df, prot_attr, target_col, num_rounds=NUM_ROUNDS_SWEEP,
                                        seed=seed, variant='full', **kwargs)
                    row = {
                        'dataset': ds_label, 'protected_attr': prot_attr,
                        'experiment_type': 'sensitivity', 'method': 'Proposed', 'variant': 'full',
                        'param_name': param_name, 'param_value': val, 'seed': seed,
                        'selected_round': res['selected_round'], 'selected_type': res['selected_type'],
                        'acc': res['final_acc'], 'bacc': res['final_bacc'], 'di1': res['final_di1'],
                        'spd': res['final_spd'], 'eod': res['final_eod'], 'aod': res['final_aod'],
                    }
                    row.update(_extended_row_fields(res))
                    append_master_row(row)
                    append_history_rows(ds_label, prot_attr, 'sensitivity', 'full',
                                         param_name, val, seed, res['round_history'])
                    mark_done(key)
            print(f'  [sensitivity] {param_name} sweep done ({len(values)} values x {len(SEEDS_SWEEP)} seeds)')

    # ---- Experiment 3: ablation (representative datasets only) ----
    if RUN_ABLATION and (ds_name, prot_attr) in ablation_targets:
        for variant in ABLATION_VARIANTS:
            for seed in SEEDS_ABLATION:
                key = make_key(ds_label, prot_attr, 'ablation', variant, None, None, seed)
                if already_done(key):
                    continue
                res = run_pipeline(df, prot_attr, target_col, num_rounds=NUM_ROUNDS_MAIN,
                                    seed=seed, variant=variant)
                row = {
                    'dataset': ds_label, 'protected_attr': prot_attr,
                    'experiment_type': 'ablation', 'method': 'Proposed', 'variant': variant,
                    'param_name': '', 'param_value': '', 'seed': seed,
                    'selected_round': res['selected_round'], 'selected_type': res['selected_type'],
                    'acc': res['final_acc'], 'bacc': res['final_bacc'], 'di1': res['final_di1'],
                    'spd': res['final_spd'], 'eod': res['final_eod'], 'aod': res['final_aod'],
                }
                row.update(_extended_row_fields(res))
                append_master_row(row)
                append_history_rows(ds_label, prot_attr, 'ablation', variant, None, None, seed, res['round_history'])
                mark_done(key)
            print(f'  [ablation] variant={variant} done ({len(SEEDS_ABLATION)} seeds)')

print('\nAll queued experiments complete (or already were, and were skipped).')
